In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
import math

# -----------------------
# 1. Data utilities
# -----------------------
class CharDataset(Dataset):
    def __init__(self, text, seq_len):
        self.text = text
        self.seq_len = seq_len
        self.chars = sorted(list(set(text)))
        self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.itos = {i: c for c, i in self.stoi.items()}
        self.vocab_size = len(self.chars)

        self.data = [self.stoi[c] for c in text]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.seq_len]
        y = self.data[idx + 1:idx + self.seq_len + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


# -----------------------
# 2. Model definition
# -----------------------
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=128, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)
        out, hidden = self.rnn(x, hidden)
        logits = self.fc(out)
        return logits, hidden


# -----------------------
# 3. Training loop
# -----------------------
def train_model(text, seq_len=50, batch_size=64, hidden_size=128,
                epochs=10, lr=1e-3, device="cuda" if torch.cuda.is_available() else "cpu"):

    dataset = CharDataset(text, seq_len)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = CharRNN(dataset.vocab_size, embed_dim=64, hidden_size=hidden_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # simple train/val split: last 10% as val
    n_total = len(dataset)
    n_val = max(1, n_total // 10)
    n_train = n_total - n_val

    train_indices = list(range(0, n_train))
    val_indices = list(range(n_train, n_total))

    def get_batch(indices):
        xs, ys = [], []
        for idx in indices:
            x, y = dataset[idx]
            xs.append(x)
            ys.append(y)
        xs = torch.stack(xs).to(device)
        ys = torch.stack(ys).to(device)
        return xs, ys

    train_losses, val_losses = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        random.shuffle(train_indices)

        for i in range(0, len(train_indices), batch_size):
            batch_idx = train_indices[i:i + batch_size]
            x, y = get_batch(batch_idx)

            optimizer.zero_grad()
            logits, _ = model(x)
            loss = criterion(logits.view(-1, dataset.vocab_size), y.view(-1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_train = epoch_loss / math.ceil(len(train_indices) / batch_size)

        # validation
        model.eval()
        with torch.no_grad():
            x_val, y_val = get_batch(val_indices)
            logits_val, _ = model(x_val)
            val_loss = criterion(logits_val.view(-1, dataset.vocab_size), y_val.view(-1)).item()

        train_losses.append(avg_train)
        val_losses.append(val_loss)
        print(f"Epoch {epoch}: train_loss={avg_train:.4f}, val_loss={val_loss:.4f}")

    return model, dataset, train_losses, val_losses


# -----------------------
# 4. Sampling with temperature
# -----------------------
def sample(model, dataset, start_text="h", length=300, temperature=1.0,
           device="cuda" if torch.cuda.is_available() else "cpu"):
    model.eval()
    chars = list(start_text)
    hidden = None
    x = torch.tensor([[dataset.stoi[c] for c in start_text]], dtype=torch.long).to(device)

    with torch.no_grad():
        logits, hidden = model(x, hidden)

        for _ in range(length):
            logits_last = logits[:, -1, :] / temperature
            probs = torch.softmax(logits_last, dim=-1)
            idx = torch.multinomial(probs, num_samples=1).item()
            c = dataset.itos[idx]
            chars.append(c)

            x = torch.tensor([[idx]], dtype=torch.long).to(device)
            logits, hidden = model(x, hidden)

    return "".join(chars)


if __name__ == "__main__":
    # tiny toy corpus
    toy_text = "hello hello help hello hell hello help " * 50

    model, dataset, train_losses, val_losses = train_model(
        toy_text,
        seq_len=50,
        batch_size=64,
        hidden_size=128,
        epochs=10
    )

    for tau in [0.7, 1.0, 1.2]:
        print(f"\n=== Sample (temperature={tau}) ===")
        print(sample(model, dataset, start_text="h", length=300, temperature=tau))


Epoch 1: train_loss=1.0990, val_loss=0.4467
Epoch 2: train_loss=0.2854, val_loss=0.2105
Epoch 3: train_loss=0.1947, val_loss=0.1805
Epoch 4: train_loss=0.1621, val_loss=0.1344
Epoch 5: train_loss=0.1077, val_loss=0.0900
Epoch 6: train_loss=0.0774, val_loss=0.0685
Epoch 7: train_loss=0.0616, val_loss=0.0554
Epoch 8: train_loss=0.0546, val_loss=0.0654
Epoch 9: train_loss=0.0530, val_loss=0.0489
Epoch 10: train_loss=0.0473, val_loss=0.0457

=== Sample (temperature=0.7) ===
hello hello help hello hell hello help hello hello help hello hell hello help hello hello help hello hell hello help hello hello help hello hell hello help hello hello help hello hell hello help hello hello help hello hell hello help hello hello help hello hell hello help hello hello help hello hell 

=== Sample (temperature=1.0) ===
hello hello help hello hello help hello help hello hell hello help hello hell hello help hello hello help hello hell hello help hello hello help hello hell hello help helloo help hello hell